In [1]:
import os
import time
import sys
from pathlib import Path

import json
import uuid

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from pyspark.sql.functions import col
from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    read_current_snapshot,
    write_clickhouse,
)
from etl.transformations.dimensions.scd.common import (
    add_hash,
    build_expired_version,
    build_new_version,
    get_initial_valid_from,
    split_changes,
)

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_sensor_type(
    sensor_type_df: DataFrame,
) -> DataFrame:
    """
    Build dim_sensor_type warehouse shape from source snapshot.
    """

    return sensor_type_df.select(
        sensor_type_df.id.alias("sensor_type_id"),
        sensor_type_df.name,
        sensor_type_df.unit,
        sensor_type_df.description,
        sensor_type_df.optimal_min,
        sensor_type_df.optimal_max,
    )


def main():
    """
    Load dim_sensor_type as an SCD2 dimension.
    """

    spark = create_spark(
        "load_dim_sensor_type",
    )

    try:
        # Read the current state of all source tables.
        #
        # MinIO contains incremental batches, so this reconstructs the
        # latest PostgreSQL state by combining all batches and keeping
        # the newest record per primary key.
        sensor_type_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "sensor_types",
        )

        # Build the complete current dimension state.
        dim_sensor_type_source_df = transform_dim_sensor_type(
            sensor_type_df,
        )

        # Read currently active SCD2 versions from ClickHouse.
        current_dim_sensor_type_df = read_clickhouse(
            spark,
            """
            (
                SELECT *
                FROM dim_sensor_type FINAL
            ) AS dim_sensor_type
            """,
        ).filter(
            col("is_current") == 1,
        )

        # Add hashes to compare business attributes.
        #
        # The hash represents the state of the dimension row.
        # If the hash changes, a new SCD2 version is created.
        source_hashed_df = add_hash(
            dim_sensor_type_source_df,
            [
                "name",
                "unit",
                "description",
                "optimal_min",
                "optimal_max",
            ],
        )

        current_hashed_df = add_hash(
            current_dim_sensor_type_df,
            [
                "name",
                "unit",
                "description",
                "optimal_min",
                "optimal_max",
            ],
        )

        # Detect new rows and changed rows.
        new_sensor_types_df, changed_sensor_types_df = split_changes(
            source_hashed_df,
            current_hashed_df,
            "sensor_type_id",
        )

        # Find currently active versions that need to expire.
        expired_sensor_types_df = current_dim_sensor_type_df.join(
            changed_sensor_types_df.select(
                "sensor_type_id",
            ),
            "sensor_type_id",
            "inner",
        )

        # Changed source rows become new active versions.
        new_sensor_types_version_df = changed_sensor_types_df

        # One version identifier for this warehouse load.
        load_version = int(
            time.time() * 1000,
        )

        # Return initial SCD2 valid_from date for the first load.

        # On initial dimension load, historical compatibility is needed
        # because fact records may exist before the warehouse dimension.
        # For subsequent loads, new versions should use current_timestamp().
        initial_valid_from = get_initial_valid_from(current_dim_sensor_type_df)

        rows_to_write = (
            build_new_version(
                new_sensor_types_df,
                load_version,
                ["sensor_type_key"],
                valid_from=initial_valid_from,
            )
            .unionByName(
                build_expired_version(
                    expired_sensor_types_df,
                    load_version,
                    ["sensor_type_key"],
                )
            )
            .unionByName(
                build_new_version(
                    new_sensor_types_version_df,
                    load_version,
                    ["sensor_type_key"],
                    valid_from=None,
                )
            )
        )

        #
        # 8. Write to ClickHouse
        if rows_to_write.count() > 0:
            write_clickhouse(
                rows_to_write,
                "dim_sensor_type",
            )

    finally:
        spark.stop()


if __name__ == "__main__":
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 09:16:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 09:16:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/07/23 09:16:37 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
